# Day 2: Pitch Distributions and Mode
**Date:** Monday 22 June 2026

**Conceptual frame:** Frequency ≠ function. Key-finding algorithms encode theories
of tonal cognition — and those theories fail on modal music in instructive ways.


```{admonition} Conceptual check — before you code
:class: tip

Answer the self-assessment questions for Day 2 before running the cells below.
Questions open in a new tab — come back here when you're done.

**[→ Open Day 2 Quiz](../quizpages/day2_quiz.md)**
```


---
## Part 1: Setup and Load


In [ ]:
import requests, zipfile
from pathlib import Path
from collections import Counter
from itertools import islice

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from music21 import converter, note, interval

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.figsize'] = (10, 4)
print('Imports OK.')

In [ ]:
CORPUS_DIR = Path('beregovski_corpus')
KERN_DIR = CORPUS_DIR / 'kern'

if KERN_DIR.exists() and len(list(KERN_DIR.glob('*.krn'))) > 0:
    print(f'Corpus ready: {len(list(KERN_DIR.glob("*.krn")))} files.')
else:
    print('Downloading corpus from GitHub...')
    CORPUS_DIR.mkdir(exist_ok=True)
    r = requests.get('https://github.com/shanahdt/mode_in_klezmer/archive/refs/heads/main.zip')
    zp = CORPUS_DIR / 'repo.zip'
    zp.write_bytes(r.content)
    import shutil
    with zipfile.ZipFile(zp) as z: z.extractall(CORPUS_DIR)
    src = list(CORPUS_DIR.glob('mode_in_klezmer-*/kern'))
    if src:
        if KERN_DIR.exists(): shutil.rmtree(KERN_DIR)
        shutil.copytree(src[0], KERN_DIR)
        zp.unlink()
    print(f'Done. {len(list(KERN_DIR.glob("*.krn")))} kern files ready.')

In [ ]:
def load_corpus(kern_dir=KERN_DIR, verbose=True):
    pc2d = {7:1,9:2,11:3,0:4,2:5,4:6,6:7,8:2,10:3,1:4,3:5,5:6}
    records, sdict = {}, {}
    files = sorted(Path(kern_dir).glob('*.krn'))
    for i, f in enumerate(files):
        if verbose and i % 50 == 0: print(f'  {i+1}/{len(files)}...')
        try:
            s = converter.parse(str(f))
            ns = [n for n in s.flat.notes if isinstance(n, note.Note)]
            pcs = [n.pitch.pitchClass for n in ns]
            records[f.stem] = {
                'tune_id': f.stem, 'n_notes': len(ns),
                'pitches': [n.nameWithOctave for n in ns],
                'pitch_classes': pcs,
                'scale_degrees': [pc2d.get(p, 0) for p in pcs],
                'intervals': [interval.Interval(ns[j], ns[j+1]).semitones
                               for j in range(len(ns)-1)]
            }
            sdict[f.stem] = s
        except: pass
    if verbose: print(f'Loaded {len(records)} tunes.')
    return pd.DataFrame(records.values()), sdict

def get_ngrams(seq, n):
    return list(zip(*[islice(seq, i, None) for i in range(n)]))

print('Helper functions defined.')

In [ ]:
print('Loading corpus...')
df, streams = load_corpus()
try:
    meta = pd.read_csv('https://raw.githubusercontent.com/shanahdt/mode_in_klezmer/main/metadata.csv')
    df = df.merge(meta, on='tune_id', how='left')
    print(f'Metadata joined. {len(df)} tunes, columns: {df.columns.tolist()}')
except Exception as e:
    print(f'Metadata not loaded ({e}). Proceeding without mode labels.')

---
## Part 2: Mode Profiles

Build a pitch class profile for each of the four klezmer modes and compare them side by side.


In [ ]:
def pc_profile_for(df_, mode=None):
    subset = df_[df_['mode']==mode] if mode and 'mode' in df_.columns else df_
    counts = [0]*12
    for pcs in subset['pitch_classes']:
        for pc in pcs: counts[pc] += 1
    total = sum(counts) or 1
    return [c/total for c in counts]

pc_names = ['C','C#','D','Eb','E','F','F#','G','Ab','A','Bb','B']
modes = [m for m in ['freygish','raised_fourth','minor','major']
         if 'mode' in df.columns and m in df['mode'].values]
mode_profiles = {m: pc_profile_for(df, m) for m in modes}

colors = {'freygish':'steelblue','raised_fourth':'coral',
          'minor':'seagreen','major':'mediumpurple'}
fig, axes = plt.subplots(2,2,figsize=(12,7),sharey=True)
for ax, mode in zip(axes.flat, modes):
    ax.bar(pc_names, mode_profiles[mode], color=colors.get(mode,'gray'),
           edgecolor='white', linewidth=0.5)
    ax.set_title(mode); ax.tick_params(axis='x',rotation=45)
fig.suptitle('Pitch class profiles by mode', fontsize=13)
plt.tight_layout(); plt.show()

---
## Part 3: Key-Finding Algorithms

Compare the Beregovski mode profiles to the Krumhansl-Kessler Western key profiles.


In [ ]:
from scipy.stats import pearsonr

ks_major = [6.35,2.23,3.48,2.33,4.38,4.09,2.52,5.19,2.39,3.66,2.29,2.88]
ks_minor = [6.33,2.68,3.52,5.38,2.60,3.97,2.69,4.89,3.30,2.80,3.36,3.06]

print(f'{'Mode':<20} {'vs KS Major':>12} {'vs KS Minor':>12}')
print('-'*46)
for mode, profile in mode_profiles.items():
    r_maj, _ = pearsonr(profile, ks_major)
    r_min, _ = pearsonr(profile, ks_minor)
    print(f'{mode:<20} {r_maj:>12.3f} {r_min:>12.3f}')

In [ ]:
print(f'{'Tune':<30} {'Detected key':<20} {'Confidence':>10}')
print('-'*62)
for tune_id in list(streams.keys())[:15]:
    try:
        k = streams[tune_id].analyze('key')
        mode_label = df[df.tune_id==tune_id]['mode'].iloc[0] if 'mode' in df.columns else '?'
        print(f'{tune_id:<30} {str(k):<20} {k.correlationCoefficient:>10.3f}  [true: {mode_label}]')
    except: pass

---
## Day 2 Exercise: Profile Divergence

```{admonition} Exercise
Plot your chosen subset's pitch class profile alongside the full-corpus profile for its mode.
Identify the two pitch classes where they diverge most.
Write two sentences: what the divergence might reflect about your subset.
```


In [ ]:
MY_MODE = 'freygish'  # <-- change to your mode

my_subset = df[df['mode']==MY_MODE] if 'mode' in df.columns else df
my_profile = pc_profile_for(my_subset)
full_profile = mode_profiles.get(MY_MODE, [0]*12)

x = range(12)
fig, ax = plt.subplots()
ax.bar([i-0.2 for i in x], full_profile, 0.35, label=f'Full {MY_MODE}',
       color='steelblue', alpha=0.8)
ax.bar([i+0.2 for i in x], my_profile, 0.35, label='My subset',
       color='coral', alpha=0.8)
ax.set_xticks(list(x)); ax.set_xticklabels(pc_names)
ax.legend(); plt.tight_layout(); plt.show()

diffs = sorted([(abs(my_profile[i]-full_profile[i]), pc_names[i]) for i in range(12)], reverse=True)
print('Largest divergences:', diffs[:3])

---
## Project Log — Entry 2

> *The pitch distribution of my subset looks like [description].*  
> *A standard key-finding algorithm performs [well/poorly] on it because [explanation].*  
> *One thing the histogram cannot tell me about my subset is...*

*(Write here — 100–150 words)*
